# ChatGLM3-6B 官方模型推理实践

本 Notebook 演示如何在 Google Colab（推荐 V100 高显存配置）或本地 GPU 环境中加载 ChatGLM3-6B 模型并进行多轮对话推理，内容涵盖：环境配置、模型加载、模型结构查看、参数统计以及模型文件持久化到云盘。

> **运行环境建议**：推荐使用 V100 + 高内存（High-RAM）配置的 Colab 实例。ChatGLM3-6B 全精度（float16）约占用 **12 GB 显存**，V100（16 GB）可完整运行；T4（16 GB）也基本满足，但速度较慢。

## 一、克隆官方仓库与查看目录结构

In [ ]:
# # 从 GitHub 克隆 ChatGLM3 官方仓库到当前工作目录（Colab 中为 /content/ChatGLM3）
# # !git clone 是 Jupyter/Colab 中执行 Shell 命令的语法，在命令前加 ! 即可
# # 克隆完成后目录中会包含：requirements.txt、推理脚本、微调脚本等
# !git clone git@github.com:zai-org/ChatGLM3.git

In [8]:
# # 列出当前目录下的文件和文件夹，确认 ChatGLM3 仓库已成功克隆
# # !ls 在 Linux/Colab 环境下输出当前目录内容，类似 Windows 的 dir 命令
# # 预期输出中应包含 "ChatGLM3" 目录
# !ls

In [9]:
# # %cd 是 IPython 魔法命令，切换当前工作目录到 ChatGLM3/，与 os.chdir() 等效
# # 注意：%cd 对整个 Notebook 会话生效，后续所有相对路径均以此为基准
# %cd ChatGLM3

# # 输出 requirements.txt 文件内容，查看项目所需的依赖库及版本约束
# # !cat 在 Linux 中打印文件全部内容到终端输出
# !cat requirements.txt

# # 此命令已注释：在全新环境中可取消注释，按 requirements.txt 安装所有依赖
# # 当前环境已手动通过下方单独的 pip install 安装了必要依赖，无需整体安装
# !pip install -r requirements.txt

## 二、安装依赖并加载ChatGLM3-6B模型与分词器

In [2]:
# !pip install transformers==4.40.0
from pathlib import Path
MODEL_PATH=Path("model/ChatGLM3")

from transformers import AutoTokenizer, AutoModel  # 导入自动分词器和自动模型类
# AutoTokenizer : 根据模型配置文件自动选择对应的分词器实现
# AutoModel     : 根据配置自动加载模型（ChatGLM3 使用 AutoModel 而非 AutoModelForCausalLM）

# 从 HuggingFace Hub 加载 ChatGLM3-6B 的分词器
# trust_remote_code=True : 允许运行模型仓库中自定义的分词器代码（ChatGLM 系列均需此参数）
# tokenizer 类型: transformers.PreTrainedTokenizer 子类实例
tokenizer = AutoTokenizer.from_pretrained("THUDM/chatglm3-6b", trust_remote_code=True,cache_dir=MODEL_PATH/"tokenizer")

# 从 HuggingFace Hub 加载 ChatGLM3-6B 的预训练模型权重
# trust_remote_code=True : 允许运行自定义模型架构代码
# device='cuda'          : 直接将模型加载并放置在 GPU 上（要求显存 ≥ 12 GB）
#                          若显存不足可改为 device_map="auto" 或使用量化版本
# model 类型: transformers.PreTrainedModel 子类实例（ChatGLMForConditionalGeneration）
model = AutoModel.from_pretrained("THUDM/chatglm3-6b", trust_remote_code=True, device='cuda',cache_dir=MODEL_PATH/"model")

# 将模型切换到评估（推理）模式
# .eval() 关闭 Dropout 层的随机丢弃，固定 BatchNorm 的统计量，确保推理结果确定性
# 返回类型: 同 model，即切换模式后的同一个模型对象（原地修改并返回自身）
model = model.eval()


d:\miniconda3\envs\chatglm\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

d:\miniconda3\envs\chatglm\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## 三、查看模型结构与参数统计

In [3]:
# 直接输入变量名，Jupyter 自动调用 __repr__ 方法展示模型完整层级结构
# 输出内容包括：各子模块名称（如 transformer、encoder、layers）、类型和嵌套层次
# 可以直观地看到 ChatGLM3-6B 的 28 层 GLMBlock、注意力头数等关键超参数
model  # 返回: ChatGLMForConditionalGeneration 对象，Jupyter 会格式化显示模型树

ChatGLMForConditionalGeneration(
  (transformer): ChatGLMModel(
    (embedding): Embedding(
      (word_embeddings): Embedding(65024, 4096)
    )
    (rotary_pos_emb): RotaryEmbedding()
    (encoder): GLMTransformer(
      (layers): ModuleList(
        (0-27): 28 x GLMBlock(
          (input_layernorm): RMSNorm()
          (self_attention): SelfAttention(
            (query_key_value): Linear(in_features=4096, out_features=4608, bias=True)
            (core_attention): CoreAttention(
              (attention_dropout): Dropout(p=0.0, inplace=False)
            )
            (dense): Linear(in_features=4096, out_features=4096, bias=False)
          )
          (post_attention_layernorm): RMSNorm()
          (mlp): MLP(
            (dense_h_to_4h): Linear(in_features=4096, out_features=27392, bias=False)
            (dense_4h_to_h): Linear(in_features=13696, out_features=4096, bias=False)
          )
        )
      )
      (final_layernorm): RMSNorm()
    )
    (output_layer): Linear(in_

In [4]:
import numpy as np  # 导入 NumPy 数值计算库（此处未直接使用，可用于后续参数统计计算）

# model.named_parameters() 返回一个迭代器，每次产出 (参数名: str, 参数张量: torch.Tensor)
# enumerate 为迭代器加上整数索引，产出 (idx: int, (key: str, value: torch.Tensor))
# 遍历输出模型中每一个可训练参数的名称和形状，便于了解模型参数分布
for idx, (key, value) in enumerate(model.named_parameters()):
    # key   类型: str，参数的层级路径名（如 "transformer.embedding.word_embeddings.weight"）
    # value 类型: torch.nn.Parameter（torch.Tensor 的子类）
    # value.shape 类型: torch.Size，表示参数张量的各维度大小（如 torch.Size([65024, 4096])）
    # f-string 中 key:^40 表示将 key 字符串居中对齐，总宽度 40 个字符，使输出整齐
    print(f"{key:^40} paramerters num:  {value.shape}")

transformer.embedding.word_embeddings.weight paramerters num:  torch.Size([65024, 4096])
transformer.encoder.layers.0.input_layernorm.weight paramerters num:  torch.Size([4096])
transformer.encoder.layers.0.self_attention.query_key_value.weight paramerters num:  torch.Size([4608, 4096])
transformer.encoder.layers.0.self_attention.query_key_value.bias paramerters num:  torch.Size([4608])
transformer.encoder.layers.0.self_attention.dense.weight paramerters num:  torch.Size([4096, 4096])
transformer.encoder.layers.0.post_attention_layernorm.weight paramerters num:  torch.Size([4096])
transformer.encoder.layers.0.mlp.dense_h_to_4h.weight paramerters num:  torch.Size([27392, 4096])
transformer.encoder.layers.0.mlp.dense_4h_to_h.weight paramerters num:  torch.Size([4096, 13696])
transformer.encoder.layers.1.input_layernorm.weight paramerters num:  torch.Size([4096])
transformer.encoder.layers.1.self_attention.query_key_value.weight paramerters num:  torch.Size([4608, 4096])
transformer.encod

## 四、单轮对话推理

In [16]:
# 此行已注释：原本从控制台读取用户输入，此处改为硬编码问题以便演示
# input() 函数在 Colab/Jupyter 中可交互运行，若需交互式对话可取消注释
# data=input("请输入:")

# model.stream_chat 是 ChatGLM3 封装的流式推理接口，逐 token 实时输出，实现"打字机"效果
# 与 model.chat 的区别：chat 等待全部生成完毕后一次性返回；stream_chat 每生成一个 token 就立即 yield
#
# 完整签名：
#   model.stream_chat(tokenizer, query, history=None, role='user',
#                     past_key_values=None, max_length=8192, do_sample=True,
#                     top_p=0.8, temperature=0.8, logits_processor=None,
#                     return_past_key_values=False, **kwargs)
#
# 参数说明：
#   tokenizer              : PreTrainedTokenizer，分词器实例，负责文本与 token 的转换
#   query                  : str，用户当前轮次的输入文本，即本次提问内容
#   history                : List[Dict]，历史对话列表，每项格式为 {"role": "user"/"assistant", "content": "..."}
#                            传入空列表表示全新对话的第一轮，不携带上下文
#   role                   : str，当前消息的角色，默认 "user"；工具调用结果时可设为 "observation"
#   past_key_values        : 上一轮的 KV Cache，传入可复用缓存、加速推理，默认 None
#   max_length             : int，生成文本的最大 token 长度（含输入），默认 8192
#   do_sample              : bool，是否随机采样，默认 True；False 时退化为贪心解码
#   top_p                  : float，核采样阈值，默认 0.8；值越小输出越保守
#   temperature            : float，温度系数，默认 0.8；<1 更保守，>1 更发散
#   logits_processor       : LogitsProcessorList，自定义 logits 处理器，默认 None
#   return_past_key_values : bool，是否在每次 yield 时同时返回 KV Cache，默认 False
#   **kwargs               : 透传给底层 model.generate()
#
# 返回值（生成器，每次 yield）：
#   response : str，截至当前 token 的累积回答文本（每次都是完整文本，非增量片段）
#   history  : List[Dict]，当前轮次更新后的对话历史
#   （若 return_past_key_values=True，则额外 yield past_key_values）

from IPython.display import display, HTML  # display 支持 display_id 原地更新；HTML 用于正确渲染换行

response = ""  # 初始化回答字符串，用于保存最终完整回答；类型: str
history = []   # 初始化对话历史列表；类型: List[Dict]

# 创建一个带 display_id 的占位输出区域
# HTML('') 初始为空 HTML，不会显示多余的 '' 符号
# display_id=True 会自动分配唯一 ID，后续可通过 handle.update() 原地更新内容，无需清屏
# handle 类型: IPython.core.display_functions.DisplayHandle
handle = display(HTML(''), display_id=True)

# 遍历流式生成器，每次迭代获得截至当前 token 的累积回答
# stream_chat 每次 yield 的 response 是「累积全文」而非增量片段
for response, history in model.stream_chat(tokenizer, 'AI太难学怎么办', history=[], max_length=200):
    # 将 response 中的换行符 \n 替换为 HTML 换行标签 <br>，确保多行文本正确显示
    # white-space: pre-wrap 保留空格和换行，同时允许自动折行
    # handle.update()：原地更新同一块输出区域，不销毁重建 DOM，彻底消除闪烁
    handle.update(HTML(f'<div style="white-space: pre-wrap">{response}</div>'))

## 五、模型文件持久化到Google云盘（可选）

In [8]:
# 挂载 Google 云盘到 Colab 虚拟机，使 Colab 可以读写 Google Drive 中的文件
# 取消注释后运行会弹出授权链接，完成授权后 Google Drive 被挂载到 /content/drive/
# 挂载后可将模型文件复制到云盘，避免 Colab 运行时重置导致重新下载（节省时间）
# from google.colab import drive        # 导入 Colab 专属的 drive 工具模块
# drive.mount('/content/drive')         # 将 Google Drive 挂载到 /content/drive 路径

In [9]:
# 查看 HuggingFace Hub 缓存目录中已下载的 ChatGLM3-6B 模型文件，并将其复制到 Google 云盘
# HuggingFace 默认将下载的模型权重缓存在 /root/.cache/huggingface/hub/ 目录下
import os  # 导入标准库 os，用于文件系统操作（listdir、system 等）

# 列出 ChatGLM3-6B 模型缓存目录中的所有文件
# /root/.cache/huggingface/hub/models--THUDM--chatglm3-6b : HuggingFace 缓存中模型文件所在路径
# os.listdir() 返回类型: list[str]，目录中所有文件和子目录的名称列表
# os.listdir('/root/.cache/huggingface/hub/models--THUDM--chatglm3-6b')

# 将整个模型缓存目录递归复制到 Google Drive 的 MyDrive 根目录
# os.system() 执行系统 Shell 命令，返回类型: int（0 表示成功，非 0 表示失败）
# cp -r : 递归复制目录及其所有子目录和文件
# 复制完成后，即使 Colab 运行时重置，下次可直接从云盘恢复模型，无需重新下载
# os.system('cp -r /root/.cache/huggingface/hub/models--THUDM--chatglm3-6b /content/drive/MyDrive/')